# Custom Insurance Cost Inference
## Manual Input Evaluation for `best_model_classic.pkl`

This notebook lets you **manually set patient values** in the input cell below,
then run the remaining cells to see the **predicted insurance charges**
alongside a breakdown of the engineered features sent to the model.

---
## 1. Setup & Load Model

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.simplefilter(action='ignore')

# Load the trained model
model = joblib.load('best_model_classic.pkl')
print(f"Model loaded: {type(model).__name__}")
print(f"Expected features ({model.n_features_in_}):")
for i, f in enumerate(model.feature_names_in_, 1):
    print(f"  {i:2d}. {f}")

Model loaded: XGBRegressor
Expected features (21):
   1. age
   2. bmi
   3. children
   4. smoker_binary
   5. smoker_bmi
   6. smoker_age
   7. age_sq
   8. bmi_sq
   9. age_bmi
  10. is_obese
  11. is_overweight
  12. smoker_obese
  13. age_group_young
  14. age_group_mid
  15. age_group_senior
  16. has_children
  17. log_bmi
  18. sex_male
  19. region_northwest
  20. region_southeast
  21. region_southwest


---
## 2. Feature Engineering Pipeline

In [2]:
def prepare_features(age, sex, bmi, children, smoker, region):
    """
    Takes raw patient info and creates the full feature vector
    matching the training pipeline.

    Parameters:
    -----------
    age      : int   — Patient age (18-64 in training data)
    sex      : str   — 'male' or 'female'
    bmi      : float — Body Mass Index (15-53 in training data)
    children : int   — Number of children (0-5 in training data)
    smoker   : str   — 'yes' or 'no'
    region   : str   — 'northeast', 'northwest', 'southeast', 'southwest'

    Returns:
    --------
    pd.DataFrame — Single-row DataFrame with all 21 engineered features
    """
    smoker_binary = 1 if smoker == 'yes' else 0

    features = {
        'age': age,
        'bmi': bmi,
        'children': children,
        'smoker_binary': smoker_binary,
        'smoker_bmi': smoker_binary * bmi,
        'smoker_age': smoker_binary * age,
        'age_sq': age ** 2,
        'bmi_sq': bmi ** 2,
        'age_bmi': age * bmi,
        'is_obese': int(bmi >= 30),
        'is_overweight': int(bmi >= 25),
        'smoker_obese': smoker_binary * int(bmi >= 30),
        'age_group_young': int(age < 30),
        'age_group_mid': int(30 <= age < 50),
        'age_group_senior': int(age >= 50),
        'has_children': int(children > 0),
        'log_bmi': np.log1p(bmi),
        'sex_male': int(sex == 'male'),
        'region_northwest': int(region == 'northwest'),
        'region_southeast': int(region == 'southeast'),
        'region_southwest': int(region == 'southwest'),
    }

    return pd.DataFrame([features])


def predict_charges(age, sex, bmi, children, smoker, region):
    """
    Predict insurance charges for a patient.
    Model was trained on log1p(charges), so we inverse-transform with expm1.
    """
    X = prepare_features(age, sex, bmi, children, smoker, region)
    y_log = model.predict(X)[0]
    charges = np.expm1(y_log)
    return max(charges, 0)  # Clip negative predictions


print("Feature pipeline ready.")
print("Usage: predict_charges(age, sex, bmi, children, smoker, region)")

Feature pipeline ready.
Usage: predict_charges(age, sex, bmi, children, smoker, region)


---
## 3. ✏️ Custom Input — *Edit the values below*

Change the variables in the next cell to whatever patient profile you want
to evaluate, then **run the remaining cells** to see the prediction.

| Variable   | Type    | Valid values / range                                  |
|------------|---------|------------------------------------------------------|
| `age`      | `int`   | 18 – 64  *(training range)*                          |
| `sex`      | `str`   | `'male'` or `'female'`                               |
| `bmi`      | `float` | 15.0 – 53.0  *(training range)*                     |
| `children` | `int`   | 0 – 5  *(training range)*                            |
| `smoker`   | `str`   | `'yes'` or `'no'`                                    |
| `region`   | `str`   | `'northeast'`, `'northwest'`, `'southeast'`, `'southwest'` |

In [3]:
# ╔══════════════════════════════════════════════════════════════╗
# ║           ✏️  EDIT YOUR CUSTOM INPUT HERE  ✏️               ║
# ╚══════════════════════════════════════════════════════════════╝

age      = 30           # Patient age
sex      = 'male'       # 'male' or 'female'
bmi      = 25.0         # Body Mass Index
children = 0            # Number of children
smoker   = 'no'         # 'yes' or 'no'
region   = 'northeast'  # 'northeast', 'northwest', 'southeast', 'southwest'

print("Custom input set ✓")
print(f"  Age      : {age}")
print(f"  Sex      : {sex}")
print(f"  BMI      : {bmi}")
print(f"  Children : {children}")
print(f"  Smoker   : {smoker}")
print(f"  Region   : {region}")

Custom input set ✓
  Age      : 30
  Sex      : male
  BMI      : 25.0
  Children : 0
  Smoker   : no
  Region   : northeast


---
## 4. Prediction Result

In [4]:
# --- Run prediction ---
predicted_charge = predict_charges(age, sex, bmi, children, smoker, region)

print("=" * 60)
print("             INSURANCE COST PREDICTION")
print("=" * 60)
print(f"  Age      : {age}")
print(f"  Sex      : {sex}")
print(f"  BMI      : {bmi}")
print(f"  Children : {children}")
print(f"  Smoker   : {smoker}")
print(f"  Region   : {region}")
print("-" * 60)
print(f"  💰 Predicted Charges : ${predicted_charge:,.2f}")
print("=" * 60)

             INSURANCE COST PREDICTION
  Age      : 30
  Sex      : male
  BMI      : 25.0
  Children : 0
  Smoker   : no
  Region   : northeast
------------------------------------------------------------
  💰 Predicted Charges : $4,111.67


---
## 5. Engineered Feature Breakdown

The table below shows every engineered feature that was actually fed
to the model for this prediction.

In [5]:
# --- Show all engineered features ---
features_df = prepare_features(age, sex, bmi, children, smoker, region)

print("Engineered features sent to the model:")
print("=" * 40)
for col in features_df.columns:
    val = features_df[col].iloc[0]
    if isinstance(val, float):
        print(f"  {col:<22s} : {val:>12.4f}")
    else:
        print(f"  {col:<22s} : {val:>12}")
print("=" * 40)

Engineered features sent to the model:
  age                    :           30
  bmi                    :      25.0000
  children               :            0
  smoker_binary          :            0
  smoker_bmi             :       0.0000
  smoker_age             :            0
  age_sq                 :          900
  bmi_sq                 :     625.0000
  age_bmi                :     750.0000
  is_obese               :            0
  is_overweight          :            1
  smoker_obese           :            0
  age_group_young        :            0
  age_group_mid          :            1
  age_group_senior       :            0
  has_children           :            0
  log_bmi                :       3.2581
  sex_male               :            1
  region_northwest       :            0
  region_southeast       :            0
  region_southwest       :            0


---
## 6. Input Validation Warnings

In [6]:
# --- Validate inputs against training data ranges ---
warnings_list = []

if age < 18 or age > 64:
    warnings_list.append(f"⚠️  Age ({age}) is outside training range [18, 64]")
if bmi < 15.96 or bmi > 53.13:
    warnings_list.append(f"⚠️  BMI ({bmi}) is outside training range [15.96, 53.13]")
if children < 0 or children > 5:
    warnings_list.append(f"⚠️  Children ({children}) is outside training range [0, 5]")
if sex not in ('male', 'female'):
    warnings_list.append(f"⚠️  Sex ('{sex}') must be 'male' or 'female'")
if smoker not in ('yes', 'no'):
    warnings_list.append(f"⚠️  Smoker ('{smoker}') must be 'yes' or 'no'")
if region not in ('northeast', 'northwest', 'southeast', 'southwest'):
    warnings_list.append(f"⚠️  Region ('{region}') must be one of: northeast, northwest, southeast, southwest")
if age < 0:
    warnings_list.append(f"❌  Age ({age}) is negative — invalid input!")
if bmi < 0:
    warnings_list.append(f"❌  BMI ({bmi}) is negative — invalid input!")

if warnings_list:
    print("INPUT VALIDATION WARNINGS")
    print("=" * 60)
    for w in warnings_list:
        print(f"  {w}")
    print("=" * 60)
    print("\nNote: Predictions for inputs outside the training range")
    print("may be unreliable due to extrapolation.")
else:
    print("✅ All inputs are within valid training data ranges.")

✅ All inputs are within valid training data ranges.
